In [ ]:
import os
import cv2 as cv
import numpy as np

In [ ]:
Z = 256
Z_max = 255
Z_min = 0
gamma = 2.2

In [ ]:
def ReadImg(path, flag=1):
  img = cv.imread(path, flag)  # flag = 1 means to load a color image
  img = img[:,:,[2,1,0]]
  return img

In [ ]:
def SaveImg(img, path):
  img = img[:,:,[2,1,0]]
  cv.imwrite(path, img)

In [ ]:
def Load_Exposures(source_dir):
  """ load bracketing images folder

  Args:
      source_dir (string): folder path containing bracketing images and a image_list.txt file
                            image_list.txt contains lines of image_file_name, exposure time, ...
  Returns:
      img_list (uint8 ndarray, shape (N, height, width, ch)): N bracketing images (3 channel)
      exposure_times (list of float, size N): N exposure times
  """
  filenames = []
  exposure_times = []
  f = open(os.path.join(source_dir, 'image_list.txt'))
  for line in f:
      if (line[0] == '#'):
          continue
      (filename, exposure, *_) = line.split()
      filenames += [filename]
      exposure_times += [float(exposure)]
  img_list = [ReadImg(os.path.join(source_dir, f)) for f in filenames]
  img_list = np.array(img_list)

  return img_list, exposure_times

In [ ]:
def Pixel_Sampling(img_list):
  """ Sampling

  Args:
      img_list (uint8 ndarray, shape (N, height, width, ch))

  Returns:
      sample (uint8 ndarray, shape (N, height_sample_size, width_sample_size, ch))
  """
  # trivial periodic sample
  sample = img_list[:, ::128, ::128, :]

  return sample

In [ ]:
def prior(z):
  """ Fit curve assumption in Equation (2)
      Data type depend on how you write your code

  Args:
      z

  Returns:
      z
  """

  ''' TODO '''
  z = np.minimum(z - Z_min, Z_max - z)

  return z

In [ ]:
def Response_Estimation(img_samples, etime_list, lambda_=50):
  """ Estimate camera response for bracketing images

  Args:
      img_samples (uint8 ndarray, shape (N, height_sample_size, width_sample_size)): N bracketing sampled images (1 channel)
      etime_list (list of float, size N): N exposure times
      lambda_ (float): Lagrange multiplier (Defaults to 50)

  Returns:
      response (float ndarray, shape (256)): response map
  """

  ''' TODO '''

  # N: Number of images, H: Height of images, W: Width of images, P: Number of pixels
  N, H, W = img_samples.shape
  P = H * W

  # Initialize matrix A nad b
  A = np.zeros((N * P + Z_max - Z_min, 256 + P))
  b = np.zeros((N * P + Z_max - Z_min))

  # Filling the data terms and exposure time
  for i in range(P):
    for j in range(N):

      h, w = i // W, i % W
      eqs = i * N + j
      z = int(img_samples[j][h][w])
      weight = prior(z)

      A[eqs][z] = weight
      A[eqs][256 + i] = -weight
      b[eqs] = weight * np.log(etime_list[j])

  # Fillig the smoothness term
  for z in range(Z_min + 1, Z_max):

    # weight = np.sqrt(lambda_) * prior(z)
    weight = lambda_ * prior(z)
    A[N * P + z - 1][z - 1] = weight
    A[N * P + z - 1][z] = -2 * weight
    A[N * P + z - 1][z + 1] = weight

  # Filling the constraint term
  A[-1][127] = 1

  # Solve the mean square problem
  x, residuals, rank, s = np.linalg.lstsq(A, b, rcond = None)
  response = x[:256]

  return response

In [ ]:
def Radiance_Construction(img_list, response, etime_list):
  """ Construct radiance map from brackting images

  Args:
      img_list (uint8 ndarray, shape (N, height, width)): N bracketing images (1 channel)
      response (float ndarray, shape (256)): response map
      etime_list (list of float, size N): N exposure times

  Returns:
      radiance (float ndarray, shape (height, width)): radiance map
  """

  ''' TODO '''

  log_etime_list = np.log(etime_list).reshape(-1, 1, 1)
  responses = response[img_list]

  w = prior(img_list)
  w_sum = np.sum(w, axis = 0)
  safe_w_sum = np.where(w_sum == 0, 1, w_sum)

  numerator = np.sum(w * (responses - log_etime_list), axis = 0)
  fallback_mean = np.mean((responses - log_etime_list), axis = 0)

  radiance = np.exp(np.where(w_sum > 0, numerator / safe_w_sum, fallback_mean))

  return radiance

In [ ]:
def Camera_Response_Calibration(src_path, lambda_):
  img_list, exposure_times = Load_Exposures(src_path)
  radiance = np.zeros_like(img_list[0], dtype=np.float32)
  pixel_samples = Pixel_Sampling(img_list)
  for ch in range(3):
      response = Response_Estimation(pixel_samples[:, :, :, ch], exposure_times, lambda_)
      radiance[:,:,ch] = Radiance_Construction(img_list[:, :, :,ch], response, exposure_times)

  return radiance

In [ ]:
def White_Balance(src, y_range, x_range):
  """ White balance adjustment based on "Known to be White" (KTBW) region

  Args:
      src (float ndarray, shape (height, width, ch)): source radiance
      y_range (tuple of 2 int): location range in y-dimension
      x_range (tuple of 2 int): location range in x-dimension

  Returns:
      result (float ndarray, shape (height, width, ch))
  """

  ''' TODO '''

  X_sum = sum(src[i][j] for i in range(y_range[0], y_range[1]) for j in range(x_range[0], x_range[1]))
  result = src * X_sum[2] / X_sum

  return result

In [ ]:
def Global_Tone_Mapping(src, scale=1.0):
    """ Global tone mapping

    Args:
        src (float ndarray, shape (height, width, ch)): source radiance image
        scale (float): scaling factor (Defaults to 1.0)

    Returns:
        result(uint8 ndarray, shape (height, width, ch)): result HDR image
    """

    ''' TODO '''

    gamma = 2.2
    X_max = np.max(src)
    X_hat = np.power(2, scale * (np.log2(src) - np.log2(X_max)) + np.log2(X_max))
    X_prime = np.power(X_hat, 1 / gamma)
    result = (255 * np.clip(X_prime, 0, 1)).astype(np.uint8)

    return result

In [ ]:
def Local_Tone_Mapping(src, imgFilter, scale=3.0):
  """ Local tone mapping

  Args:
      src (float ndarray, shape (height, width, ch)): source radiance image
      imgFilter (function): filter function with preset parameters
      scale (float): scaling factor (Defaults to 3.0)

  Returns:
      result(uint8 ndarray, shape (height, width, ch)): result HDR image
  """

  ''' TODO '''

  gamma = 2.2

  I = np.mean(src, axis = 2)
  C_x = src / I[:, :, np.newaxis]
  L = np.log2(I)
  L_B = imgFilter(L)
  L_D = L - L_B
  L_max = np.max(L_B)
  L_min = np.min(L_B)
  L_B_prime = (L_B - L_max) * scale / (L_max - L_min)
  I_prime = 2 ** (L_B_prime + L_D)
  C = C_x * np.stack([I_prime] * 3, axis = -1)
  result = (255 * np.clip(np.power(C, 1 / gamma), 0, 1)).astype(np.uint8)

  return result

In [ ]:
def Gaussian_Filter(src, N=35, sigma_s=100):
  """ Gaussian filter for smoothing

  Args:
      src (float ndarray, shape (height, width)): source intensity
      N (int): window size of the filter (Defaults to 35)
                filter indices span [-N/2, N/2]
      sigma_s (float): standard deviation of Gaussian filter (Defaults to 100)

  Returns:
      result (float ndarray, shape (height, width))
  """

  ''' TODO '''

  H, W = src.shape
  result = np.empty((H, W))
  pad_src = np.pad(src, pad_width = N//2, mode = "symmetric")

  ax = np.arange(-N//2, N//2)
  k, l = np.meshgrid(ax, ax)
  kernel = np.exp(-(k**2 + l**2) / (2 * sigma_s**2))
  kernel /= np.sum(kernel)

  for i in range(H):
    for j in range(W):

      result[i][j] = np.sum(pad_src[i : i + N, j : j + N] * kernel)

  return result

In [ ]:
def Bilateral_Filter(src, N=35, sigma_s=100, sigma_r=0.8):
  """ Bilateral filter for smoothing

  Args:
      src (float ndarray, shape (height, width)): source intensity
      N (int): window size of the filter (Defaults to 35)
                filter indices span [-N/2, N/2]
      sigma_s (float): spatial standard deviation of bilateral filter (Defaults to 100)
      sigma_r (float): range standard deviation of bilateral filter (Defaults to 0.8)

  Returns:
      result (float ndarray, shape (height, width))
  """

  ''' TODO '''

  H, W = src.shape
  result = np.empty((H, W))
  pad_src = np.pad(src, pad_width = N//2, mode = "symmetric")

  ax = np.arange(-N//2, N//2)
  k, l = np.meshgrid(ax, ax)
  kernel = np.exp(-(k**2 + l**2) / (2 * sigma_s**2))

  for i in range(H):
    for j in range(W):

      patch = pad_src[i : i + N, j : j + N]
      bilateral_kernel = np.exp(-((patch[N//2, N//2] - patch)**2 / (2 * sigma_r**2)))
      bilateral_kernel *= kernel
      bilateral_kernel /= np.sum(bilateral_kernel)

      result[i][j] = np.sum(patch * bilateral_kernel)

  return result

In [ ]:
# Generating pixel values v.s. log exposure time for memorial image set.
import matplotlib.pyplot as plt

def Plot_Pixel_Value_vs_Log_Exposure(src_path, lambda_ = 50):

  img_list, exposure_times = Load_Exposures(src_path)
  log_exposure_times = np.log(exposure_times)

  # Picking 3 points in the image list
  POS = [(34, 85), (2, 410), (405, 396)]

  # Picking exposure time
  IDX = [i * 3 for i in range(len(log_exposure_times) // 3)]
  LOG_TIME = [log_exposure_times[i] for i in IDX]

  # Deriving radiance
  radiance = Camera_Response_Calibration(src_path = src_path, lambda_ = lambda_)

  # Plotting points
  fig, (ax1, ax2) = plt.subplots(1, 2)
  fig.set_size_inches(10, 5)

  plt.xlim(-8, 8)
  plt.ylim(-10, 265)
  ax1.set_title(f"Pixel Value vs log Exposure Time")
  ax2.set_title(f"Pixel Value vs log Exposure Time")
  ax1.set_xlabel("log Exposure Time")
  ax1.set_ylabel("Pixel Value")
  ax2.set_xlabel("log Exposure Time + log Radiance")
  ax2.set_ylabel("Pixel Value")

  for i in range(3):
    x, y = POS[i]
    arr = np.array([np.mean(img_list[j][x][y]) for j in IDX])
    ax1.plot(LOG_TIME, arr, marker = 'o')
    ax2.plot(LOG_TIME + np.mean(np.log(radiance[x][y])), arr, marker = 'o')

  plt.show()


In [ ]:
# Plot response curve
def Plot_Response_Curve(src_path):

  img_list, exposure_times = Load_Exposures(src_path)
  lambda_list = [0, 0.5, 5, 50]
  pixel_samples = Pixel_Sampling(img_list)

  colors = ['r', 'g', 'b']
  labels = ["Red", "Green", "Blue"]
  fig, axes = plt.subplots(1, 4, figsize = (20, 5))

  for i, lambda_ in enumerate(lambda_list):
    ax = axes[i]
    for ch in range(3):
      response = Response_Estimation(pixel_samples[:, :, :, ch], exposure_times, lambda_)
      ax.plot(response, range(256), color = colors[ch], label = labels[ch])

    ax.set_title(rf"Camera Response ($\lambda = {lambda_}$)")
    ax.set_xlabel("log Exposure")
    ax.set_ylabel("Pixel Value")

  plt.tight_layout()
  plt.show()

In [ ]:
def CLAHE_Global_Tone_Mapping(src, scale=1.0):

  gamma = 2.2
  X_max = np.max(src)
  X_hat = np.power(2, scale * (np.log2(src) - np.log2(X_max)) + np.log2(X_max))
  X_prime = np.power(X_hat, 1 / gamma)

  X_percentile = np.percentile(X_prime, 97.5)
  X_norm = np.clip((X_prime / X_percentile) * 255, 0, 255).astype(np.uint8)

  lab = cv.cvtColor(X_norm, cv.COLOR_RGB2LAB)
  l, a, b = cv.split(lab)

  clahe = cv.createCLAHE(clipLimit = 2.5, tileGridSize = (8, 8))
  l_clahe = clahe.apply(l)

  l_ratio = l_clahe.astype(np.float32) / l.astype(np.float32)
  l_ratio = np.clip(l_ratio, 1, 2.5)

  a = np.clip((a.astype(np.float32) - 128.0) * l_ratio + 128.0, 0, 255).astype(np.uint8)
  b = np.clip((b.astype(np.float32) - 128.0) * l_ratio + 128.0, 0, 255).astype(np.uint8)

  lab_clahe = cv.merge((l_clahe, a, b))
  result = cv.cvtColor(lab_clahe, cv.COLOR_LAB2RGB)

  return result

In [ ]:
def CLAHE_Local_Tone_Mapping(src, imgFilter, scale=3.0):

  gamma = 2.2

  I = np.mean(src, axis = 2)
  C_x = src / I[:, :, np.newaxis]
  L = np.log2(I)
  L_B = imgFilter(L)
  L_D = L - L_B
  L_max = np.max(L_B)
  L_min = np.min(L_B)
  L_B_prime = (L_B - L_max) * scale / (L_max - L_min)
  I_prime = 2 ** (L_B_prime + L_D)
  C = C_x * np.stack([I_prime] * 3, axis = -1)
  C_gamma = np.power(C, 1 / gamma)

  C_percentile = np.percentile(C_gamma, 99.5)
  C_norm = np.clip((C_gamma / C_percentile) * 255, 0, 255).astype(np.uint8)

  lab = cv.cvtColor(C_norm, cv.COLOR_RGB2LAB)
  l, a, b = cv.split(lab)

  clahe = cv.createCLAHE(clipLimit = 2.5, tileGridSize = (8, 8))
  l_clahe = clahe.apply(l)

  l_ratio = l_clahe.astype(np.float32) / l.astype(np.float32)
  l_ratio = np.clip(l_ratio, 1, 2.5)

  a = np.clip((a.astype(np.float32) - 128.0) * l_ratio + 128.0, 0, 255).astype(np.uint8)
  b = np.clip((b.astype(np.float32) - 128.0) * l_ratio + 128.0, 0, 255).astype(np.uint8)

  lab_clahe = cv.merge((l_clahe, a, b))
  result = cv.cvtColor(lab_clahe, cv.COLOR_LAB2RGB)

  return result

In [ ]:
def Threshold_Radiance_Construction(img_list, response, etime_list):

  log_etime_list = np.log(etime_list).reshape(-1, 1, 1)
  responses = response[img_list]

  w = prior(img_list)

  # Threshold
  threshold = 0.4
  ref_idx = 7
  log_E = responses - log_etime_list
  log_E_ref = log_E[ref_idx]

  diff = np.abs(log_E - log_E_ref)
  w[diff > threshold] = 0

  w_sum = np.sum(w, axis = 0)
  safe_w_sum = np.where(w_sum == 0, 1, w_sum)

  numerator = np.sum(w * (responses - log_etime_list), axis = 0)
  fallback_mean = np.mean((responses - log_etime_list), axis = 0)

  radiance = np.exp(np.where(w_sum > 0, numerator / safe_w_sum, fallback_mean))

  return radiance

In [ ]:
def Threshold_Camera_Response_Calibration(src_path, lambda_):
  img_list, exposure_times = Load_Exposures(src_path)
  radiance = np.zeros_like(img_list[0], dtype=np.float32)
  pixel_samples = Pixel_Sampling(img_list)
  for ch in range(3):
      response = Response_Estimation(pixel_samples[:, :, :, ch], exposure_times, lambda_)
      radiance[:,:,ch] = Threshold_Radiance_Construction(img_list[:, :, :,ch], response, exposure_times)

  return radiance